In [25]:
# Load all the Data Ingestion Steps 
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_classic.prompts import PromptTemplate

from langchain_classic.chains import RetrievalQA


In [26]:
## Read the Pdfs from the folder
loader = PyPDFDirectoryLoader("./us_census")

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

final_documents =  text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.\n1 Public policy changes included \nthe renewal of the Public Health Emergency, 

In [27]:
len(final_documents)

316

In [30]:
## Embedding using Hugging Face
huggingface_embeddings=HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",      #sentence-transformers/all-MiniLM-l6-v2
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True}

)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3358.67it/s]


In [33]:
import numpy as np
np = np.array(huggingface_embeddings.embed_query(final_documents[0].page_content))

np

array([-0.07903484, -0.01134114, -0.02312095,  0.02844461,  0.05053344,
        0.05317823, -0.01907793,  0.03456026, -0.10211366, -0.029157  ,
        0.08524264,  0.05650726, -0.02545439, -0.0330849 , -0.00635731,
        0.04090862, -0.00628106,  0.00356742, -0.03854128,  0.03667682,
       -0.04289803,  0.03425251, -0.03116898, -0.03793728,  0.0172839 ,
        0.01214924,  0.0065312 ,  0.01463564, -0.05529056, -0.15320708,
        0.00730849,  0.03202941, -0.04701126, -0.01595975,  0.01874444,
        0.02642938, -0.02306376,  0.08438038,  0.04182489,  0.05278175,
       -0.03057603,  0.0156426 , -0.01689075,  0.00529408, -0.02417433,
        0.00412996, -0.01889935, -0.00150625, -0.00836943, -0.03390062,
        0.03515958, -0.00553131,  0.04910934,  0.05971857,  0.05615962,
       -0.05105155,  0.01475134, -0.0184996 , -0.03284642,  0.03576626,
        0.04947701, -0.00938881, -0.26202112,  0.09750328,  0.01715693,
        0.04781388, -0.00556318, -0.00298309, -0.02207357, -0.04

In [34]:
# vector Store Creation
vectorstore = FAISS.from_documents(final_documents[:120],huggingface_embeddings)


In [36]:
# Query using Similarity Search
query = "WHAT IS HEALTH INSURANCE COMPANY"
relevant_documents = vectorstore.similarity_search(query)\

print(relevant_documents[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [38]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002776AC73230> search_kwargs={'k': 3}


In [40]:
from dotenv import load_dotenv
load_dotenv()

True

In [57]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage

llm = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct")
chat_model = ChatHuggingFace(llm=llm)

response = chat_model.invoke([HumanMessage(content="What is health insurance coverage?")])
print(response.content)


Health insurance coverage refers to the financial protection provided by an insurance policy designed to cover the costs associated with medical treatments, procedures, and services. This coverage can include visits to healthcare providers, prescription medications, hospital stays, and other healthcare-related expenses. Here are some key aspects of health insurance coverage:

1. **Services Covered**: The types of medical services covered can vary widely depending on the specific plan. Typically, these include doctor visits, emergency care, prescription drugs, laboratory tests, surgeries, and mental health services.

2. **Network Providers**: Many health insurance plans have a network of healthcare providers and facilities that are contracted to provide services at reduced rates for plan members. Out-of-network services may be covered, but at a higher cost.

3. **Deductibles**: This is the amount you must pay out-of-pocket before your insurance coverage kicks in. This can vary significa

In [1]:
import os
from langchain_huggingface.llms import HuggingFacePipeline

# 1. (Optional) Set token if using a gated model like Mistral or Llama
# os.environ["HF_TOKEN"] = "your_huggingface_token"

# 2. Use a model that fits on local hardware and does not require gated approvals
hf = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",  # 1.5B parameters is perfect for local CPU/GPU testing
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,         # Note: Do not set to absolute 0; use a low float like 0.1
        "max_new_tokens": 300,
        "do_sample": True           # Required by transformers when temperature is specified
    }
)

query = "What is health insurance coverage?"
response = hf.invoke(query)
print(response)


c:\Users\Abhijeet\Desktop\ai-engineer\langchain\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1902.96it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokeniz

What is health insurance coverage? Health insurance coverage refers to the benefits and services that are covered under a health insurance policy. These can include medical expenses, hospital stays, doctor visits, prescription drugs, mental health services, and more. The specific types of coverage offered by an insurance plan will depend on the type of plan you have (such as a traditional PPO or HMO) and your individual needs.

Health insurance coverage helps protect individuals from unexpected medical bills and financial burden when they need healthcare services. It provides financial protection against high out-of-pocket costs for essential medical treatments and services. By covering these expenses, health insurance helps ensure that people can access necessary care without facing significant financial hardship. 

In summary, health insurance coverage is designed to provide financial assistance in case of unforeseen medical expenses, ensuring that individuals can receive necessary m

In [ ]:
#Hugging Face models can be run locally through the HuggingFacePipeline class.
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf 
llm.invoke(query)
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])
retrievalQA=RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt":prompt}
)
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

C:\Users\Abhijeet\AppData\Local\Temp\ipykernel_3208\2872305226.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]